# EDA — Arena of Glory Spring 2026

**Đơn vị quan sát:** một trận (game). EDA chỉ mô tả dữ liệu, không fit bộ mã hóa, không huấn luyện mô hình, không dùng kết quả test để quyết định tiền xử lý. Chạy `Run All` từ thư mục dự án hoặc từ `notebook/`; cần `pandas`, `numpy`, `matplotlib`, `IPython`. Đường dẫn được tìm theo thư mục hiện hành, không cần đường dẫn tuyệt đối.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base.parent) if (p / "dataset/035ce761e53a47db/feature_schema.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Chạy notebook từ thư mục dự án hoặc notebook/ của dự án")

DATA = ROOT / "dataset/035ce761e53a47db"
CSV = DATA / "csv"
SPLITS = ("train", "validation", "test")

required = [DATA / "feature_schema.json", DATA / "split_manifest.json"] + [CSV / f"{n}.csv" for n in (*SPLITS, "games", "series", "teams", "heroes", "association_train")]
missing_files = [str(p) for p in required if not p.is_file()]
if missing_files:
    raise FileNotFoundError("Thiếu tệp: " + ", ".join(missing_files))

schema = json.loads((DATA / "feature_schema.json").read_text(encoding="utf-8"))
manifest = json.loads((DATA / "split_manifest.json").read_text(encoding="utf-8"))

frames = {s: pd.read_csv(CSV / f"{s}.csv", dtype="string") for s in SPLITS}
refs = {n: pd.read_csv(CSV / f"{n}.csv", dtype="string") for n in ("games", "series", "teams", "heroes")}
association_train = pd.read_csv(CSV / "association_train.csv", dtype="string")

features = list(schema["features"])
label = schema["label"]
hero_names = refs["heroes"].drop_duplicates("hero_id").set_index("hero_id")["name"].to_dict()
hero_ids = set(hero_names)
all_rows = pd.concat(frames, names=["tập", "dòng"])

print("Dataset:", DATA.relative_to(ROOT), "| schema:", schema["scope"])
print("Bảng tra cứu:", {k: len(v) for k, v in refs.items()}, "| giao dịch train:", len(association_train))
display(pd.DataFrame({"tập": list(frames), "số trận": [len(f) for f in frames.values()],
                      "số cột": [len(f.columns) for f in frames.values()],
                      "series duy nhất": [f["series_id"].nunique() for f in frames.values()]}))

Dataset: dataset/035ce761e53a47db | schema: post_draft_baseline_schedule_proxy
Bảng tra cứu: {'games': 299, 'series': 75, 'teams': 8, 'heroes': 86} | giao dịch train: 462


,tập,số trận,số cột,series duy nhất
0,train,231,21,60
1,validation,28,21,7
2,test,40,21,8


## 1. Cấu trúc và từ điển thuộc tính

Đọc CSV dưới dạng chuỗi để không làm mất ID hay vô tình ép nhãn. Missing gồm null và chuỗi chỉ chứa khoảng trắng; tỷ lệ tính trên số dòng của từng tập. Giá trị mẫu được cắt ngắn để tránh hiển thị nguyên provenance dài.

In [ ]:
def missing_mask(df):
    return df.isna() | df.fillna("").astype("string").apply(lambda c: c.str.strip().eq(""))

column_rows = []
for split, df in frames.items():
    miss = missing_mask(df)
    for col in df:
        nonempty = df.loc[~miss[col], col]
        column_rows.append({"tập": split, "cột": col, "dtype (khi đọc)": str(df[col].dtype),
                            "giá trị phân biệt": nonempty.nunique(),
                            "ví dụ": str(nonempty.iloc[0])[:85] if len(nonempty) else "—",
                            "thiếu": int(miss[col].sum()), "thiếu (%)": round(miss[col].mean()*100, 2)})
column_profile = pd.DataFrame(column_rows)
display(column_profile)

descriptions = {
    "game_id":"Định danh trận", "series_id":"Định danh series",
    "game_number":"Thứ tự trận trong series", "split":"Tên tập dữ liệu",
    "time_basis":"Cơ sở thời gian chia tập", "split_time":"Mốc thời gian chia tập (lịch series)",
    "team_a_heroes":"Tập 5 ID tướng đội A", "team_b_heroes":"Tập 5 ID tướng đội B",
    "team_a_side":"Phe của đội A", "patch_id":"Bản cập nhật khu vực",
    "team_a_win":"Đội A thắng hay không", "source":"Nguồn dữ liệu",
    "wiki":"Wiki nguồn", "page_id":"ID trang nguồn", "page_title":"Tên trang nguồn",
    "source_url":"URL trang nguồn", "revision_id":"ID bản sửa đổi",
    "content_hash":"Hash nội dung", "locator":"Vị trí trong trang nguồn",
    "template_hash":"Hash mẫu nguồn", "parser_version":"Phiên bản bộ phân tích"
}

all_columns = list(dict.fromkeys(c for df in frames.values() for c in df))

attribute_dictionary = pd.DataFrame([{
    "cột": c, "ý nghĩa": descriptions.get(c, "Metadata nguồn; cần xác minh"),
    "kiểu lưu trữ": "string (CSV)",
    "vai trò": "đầu vào" if c in features else "nhãn" if c == label else "metadata — không dùng dự đoán",
    "biểu diễn": schema.get("representation", {}).get(c, "chuỗi/ID; chỉ tra cứu, kiểm tra"),
} for c in all_columns])
display(attribute_dictionary)

expected = set(features + [label])
schema_check = pd.DataFrame([{"tập": s, "thiếu đầu vào/nhãn": sorted(expected - set(df)),
                               "metadata/ngoài schema": sorted(set(df) - expected),
                               "khác metadata manifest": sorted(set(df) - expected - set(manifest.get("metadata_not_features", [])))}
                              for s, df in frames.items()])
display(schema_check)
if any(schema_check["thiếu đầu vào/nhãn"].map(bool)):
    raise ValueError("Thiếu cột bắt buộc theo feature_schema.json")
print("Các cột ngoài đầu vào/nhãn chỉ dùng để kiểm tra, KHÔNG đưa vào mô hình.")

## 2–3. Toàn vẹn, missing và đội hình

Mỗi cờ lỗi tính **số dòng có lỗi**, không phải số lần xuất hiện. Lỗi có thể chồng lấn; tổng các cột lỗi không phải tổng dòng lỗi. JSON không parse được và JSON parse được nhưng không phải mảng được đếm riêng; trường null/rỗng không tính là JSON lỗi. Đội hình hợp lệ khi mỗi đội là mảng 5 ID chuỗi không rỗng, không trùng trong/giữa đội và mọi ID có trong `heroes.csv`.

In [ ]:
PATCH_PATTERN = re.compile(r"^aov:vn:\d{4}-\d{2}-\d{2}$")

def inspect_lineup(value):
    flags = {k: False for k in ("rỗng", "json_lỗi", "không_phải_mảng", "sai_số_lượng",
                                  "id_không_hợp_lệ", "lặp_trong_đội", "id_không_ánh_xạ")}
    if pd.isna(value) or not str(value).strip():
        flags["rỗng"] = True
        return [], flags
    try:
        ids = json.loads(value)
    except (ValueError, TypeError):
        flags["json_lỗi"] = True
        return [], flags
    if not isinstance(ids, list):
        flags["không_phải_mảng"] = True
        return [], flags
    flags["sai_số_lượng"] = len(ids) != 5
    flags["id_không_hợp_lệ"] = any(not isinstance(x, str) or not x.strip() for x in ids)
    valid_ids = [x for x in ids if isinstance(x, str) and x.strip()]
    flags["lặp_trong_đội"] = len(valid_ids) != len(set(valid_ids))
    flags["id_không_ánh_xạ"] = any(x not in hero_ids for x in valid_ids)
    return valid_ids, flags

parsed = {}
quality_flags = {}
for split, df in {**frames, "toàn bộ": all_rows.reset_index(drop=True)}.items():
    df = df.reset_index(drop=True)
    miss = missing_mask(df)
    team_a, team_b, issues = [], [], []
    for a, b in zip(df["team_a_heroes"], df["team_b_heroes"]):
        ids_a, fa = inspect_lineup(a)
        ids_b, fb = inspect_lineup(b)
        team_a.append(ids_a)
        team_b.append(ids_b)
        issues.append({f"{side}_{key}": value for side, flags in (("A", fa), ("B", fb))
                       for key, value in flags.items()} | {"trùng_giữa_hai_đội": bool(set(ids_a) & set(ids_b))})
    flags = pd.DataFrame(issues).fillna(False).astype(bool)
    flags["đội_hình_không_hợp_lệ"] = flags.any(axis=1)
    flags["dòng_trùng"] = df.duplicated(keep=False)
    flags["game_id_trùng"] = df["game_id"].duplicated(keep=False) & ~miss["game_id"]
    flags["game_id_thiếu"] = miss["game_id"]
    flags["nhãn_thiếu"] = miss[label]
    flags["nhãn_sai"] = ~miss[label] & ~df[label].str.lower().isin(["true", "false"])
    flags["phe_thiếu"] = miss["team_a_side"]
    flags["phe_sai"] = ~miss["team_a_side"] & ~df["team_a_side"].isin(["blue", "red"])
    flags["patch_thiếu"] = miss["patch_id"]
    flags["patch_sai_định_dạng"] = ~miss["patch_id"] & ~df["patch_id"].str.fullmatch(PATCH_PATTERN.pattern).fillna(False)
    flags["thiếu_đầu_vào"] = miss[features].any(axis=1)
    flags["có_bất_thường"] = flags.drop(columns="thiếu_đầu_vào").any(axis=1) | flags["thiếu_đầu_vào"]
    quality_flags[split] = flags
    parsed[split] = pd.DataFrame({"A": team_a, "B": team_b})

quality_table = pd.DataFrame({s: flags.sum().astype(int) for s, flags in quality_flags.items()})
quality_table.index.name = "loại lỗi (số dòng)"
display(quality_table)
missing_table = column_profile[["tập", "cột", "thiếu", "thiếu (%)"]]
display(missing_table)
display(pd.DataFrame([{"tập": s, "dòng thiếu >=1 đầu vào": int(quality_flags[s]["thiếu_đầu_vào"].sum()),
                       "tỷ lệ (%)": round(100*quality_flags[s]["thiếu_đầu_vào"].mean(), 2)} for s in SPLITS]))
# Cờ trùng toàn bộ ở trên còn bắt các trận lặp xuyên tập; kiểm tra ranh giới bằng tập ID.
for s in SPLITS:
    own = set(frames[s]["game_id"].dropna())
    print(s, "game_id chung tập khác:", len(own & set(pd.concat([frames[t]["game_id"] for t in SPLITS if t != s]).dropna())))

## 4. Nhãn, phe và patch

Kiểm tra manifest theo từng tập. Nhãn không hợp lệ không được ngầm đổi thành thua. Các bảng chéo chỉ tính nhãn hợp lệ; mỗi ô hiển thị tỷ lệ thắng quan sát và **n**. Nhóm `n < 10` được đánh dấu ít mẫu (ngưỡng hiển thị, không dùng để chọn đặc trưng).

In [ ]:
def valid_labels(df):
    return df[label].str.lower().map({"true": 1, "false": 0}).astype("Float64")

labels = {s: valid_labels(df) for s, df in frames.items()}
label_rows = []
for s, df in frames.items():
    y = labels[s]
    actual = {"games": len(df), "series": df["series_id"].nunique(),
              "true": int((y == 1).sum()), "false": int((y == 0).sum())}
    target = manifest["split_counts"][s]
    same = actual == {"games": target["games"], "series": target["series"], **target["labels"]}
    label_rows.append({"tập":s, "trận":actual["games"], "series":actual["series"],
                       "A thắng":actual["true"], "A thua":actual["false"],
                       "nhãn lỗi/thiếu":int(y.isna().sum()),
                       "A thắng (%)":round(100*actual["true"]/y.notna().sum(),2) if y.notna().sum() else np.nan,
                       "khớp manifest":same})
label_table = pd.DataFrame(label_rows).set_index("tập")
display(label_table)
if not label_table["khớp manifest"].all():
    display(Markdown("**CẢNH BÁO:** CSV khác `split_manifest.json`; cần điều tra trước khi kết luận."))

categorical_rows = []
for s, df in frames.items():
    for col in ("team_a_side", "patch_id"):
        for value, n in df[col].fillna("(thiếu)").value_counts(dropna=False).items():
            categorical_rows.append({"tập":s, "cột":col, "giá trị":value,
                                     "số trận":int(n), "tỷ lệ (%)":round(100*n/len(df),2),
                                     "ít mẫu (<10)":n<10})
categorical_table = pd.DataFrame(categorical_rows)
display(categorical_table)

valid_parts = []
for s, df in frames.items():
    part = df[["team_a_side", "patch_id"]].copy()
    part["tập"] = s
    part["A thắng"] = labels[s]
    valid_parts.append(part)
valid_games = pd.concat(valid_parts, ignore_index=True).dropna(subset=["A thắng"])
def win_cross(columns):
    result = valid_games.groupby(columns, dropna=False)["A thắng"].agg(["size", "mean"]).reset_index()
    result = result.rename(columns={"size":"n", "mean":"A thắng (%)"})
    result["A thắng (%)"] = (result["A thắng (%)"] * 100).round(1)
    result["ít mẫu (<10)"] = result["n"] < 10
    return result
side_cross = win_cross(["tập", "team_a_side"])
patch_cross = win_cross(["tập", "patch_id"])
side_patch_cross = win_cross(["tập", "team_a_side", "patch_id"])
display(side_cross, patch_cross, side_patch_cross)

## 5. Tần suất và tỷ lệ thắng quan sát theo tướng

Một lượt chọn = một tướng của một đội trong một trận. Tỷ lệ thắng tính theo **đội đã chọn tướng** (A thắng nếu nhãn `true`, B thắng nếu `false`); bảng loại đội hình không hợp lệ khỏi mẫu số và ghi số trận. Không suy ra sức mạnh/nhân quả từ tỷ lệ thô, đặc biệt khi ít mẫu. Không fit association rules trên toàn giải; bảng giao dịch chỉ được nạp từ `association_train.csv` để xác nhận nguồn train-only.

In [ ]:
picks = []
for s, df in frames.items():
    for i, row in df.reset_index(drop=True).iterrows():
        if quality_flags[s].loc[i, "đội_hình_không_hợp_lệ"]:
            continue
        y = labels[s].iloc[i]
        for team in ("A", "B"):
            for hero_id in parsed[s].at[i, team]:
                picks.append({"tập":s, "game_id":row["game_id"], "đội":team,
                              "phe":row["team_a_side"] if team=="A" else
                                     {"blue":"red", "red":"blue"}.get(row["team_a_side"], pd.NA),
                              "patch":row["patch_id"], "hero_id":hero_id,
                              "tên tướng":hero_names.get(hero_id, hero_id),
                              "đội thắng":(y if team=="A" else 1-y) if pd.notna(y) else np.nan})
picks = pd.DataFrame(picks)
train_heroes = set(picks.loc[picks["tập"]=="train", "hero_id"])
print("Tướng duy nhất toàn giải:", picks["hero_id"].nunique(), "| train:", len(train_heroes))
hero_frequency = (picks.groupby(["hero_id", "tên tướng"], as_index=False).agg(
    lượt_chọn=("game_id", "size"), số_trận=("game_id", "nunique"))
    .sort_values(["lượt_chọn", "tên tướng"], ascending=[False, True]))
display(Markdown("**Tướng được chọn nhiều nhất / ít nhất**"))
display(hero_frequency.head(15), hero_frequency.tail(15))
hero_by_context = (picks.groupby(["tập", "hero_id", "tên tướng", "đội", "phe", "patch"], dropna=False)
                   .size().rename("lượt_chọn").reset_index())
display(Markdown("**Phân tầng mẫu (chỉ xem khi đủ số trận):**"))
display(hero_by_context.sort_values("lượt_chọn", ascending=False).head(20))
hero_win = (picks.dropna(subset=["đội thắng"]).groupby(["hero_id", "tên tướng"], as_index=False)
            .agg(số_trận=("game_id", "nunique"), lượt_chọn=("game_id", "size"), thắng=("đội thắng", "sum")))
hero_win["tỷ lệ thắng đội chọn (%)"] = (100 * hero_win["thắng"] / hero_win["lượt_chọn"]).round(1)
hero_win["ít mẫu (<10 trận)"] = hero_win["số_trận"] < 10
display(hero_win.sort_values("lượt_chọn", ascending=False).head(20))
print("association_train chỉ có tập:", association_train["split"].dropna().unique().tolist())
if set(association_train["split"].dropna()) != {"train"}:
    display(Markdown("**CẢNH BÁO:** association_train.csv chứa giao dịch ngoài train!"))

## 6. So sánh các tập và danh mục mới

Danh mục mới = xuất hiện ở validation/test nhưng chưa xuất hiện ở train, **không phải missing**. So sánh này chỉ nhằm mô tả; không dùng nhãn/test để chọn cách mã hóa hoặc tiền xử lý.

In [ ]:
train_patches = set(frames["train"]["patch_id"].dropna())
new_categories = {}
compare_rows = []
for s, df in frames.items():
    used = set(picks.loc[picks["tập"]==s, "hero_id"])
    new_h = used - train_heroes
    new_p = set(df["patch_id"].dropna()) - train_patches
    new_categories[s] = {"tướng mới": sorted(new_h), "patch mới": sorted(new_p)}
    manifest_new = manifest.get("holdout_categories_not_in_train", {}).get(s, {})
    if s != "train" and (set(manifest_new.get("heroes", [])) != new_h or set(manifest_new.get("patches", [])) != new_p):
        print("CẢNH BÁO: danh mục mới khác manifest:", s)
    compare_rows.append({"tập":s, "trận":len(df), "A thắng (%)":label_table.loc[s, "A thắng (%)"],
        "blue / red (A)":f"{(df['team_a_side']=='blue').sum()} / {(df['team_a_side']=='red').sum()}",
        "patch (số trận)":str(df["patch_id"].value_counts().to_dict()),
        "tướng duy nhất":len(used), "tướng mới so train":len(new_h),
        "patch mới so train":len(new_p),
        "đội hình hợp lệ (%)":round(100*(~quality_flags[s]["đội_hình_không_hợp_lệ"]).mean(),2),
        "thiếu đầu vào":int(quality_flags[s]["thiếu_đầu_vào"].sum())})
comparison = pd.DataFrame(compare_rows).set_index("tập")
display(comparison)
display(pd.DataFrame(new_categories).T)
print("Patch chỉ có test:", sorted(set(frames["test"]["patch_id"].dropna()) - set(pd.concat([frames[s]["patch_id"] for s in ("train","validation")]).dropna())))

## 7. Trực quan hóa (6 biểu đồ)

Cỡ mẫu được ghi ở biểu đồ; `n < 10` là cảnh báo mô tả, không phải ngưỡng thống kê hay quy tắc chọn mô hình. Biểu đồ chất lượng dùng số **dòng**; các nhóm lỗi có thể chồng lấn.

In [ ]:
plt.rcParams.update({"figure.dpi":110, "axes.spines.top":False, "axes.spines.right":False})
order = list(SPLITS)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
wins = label_table.loc[order, "A thắng"].to_numpy()
losses = label_table.loc[order, "A thua"].to_numpy()
ax[0].bar(order, wins, label="A thắng", color="#258a71")
ax[0].bar(order, losses, bottom=wins, label="A thua", color="#cc6c62")
ax[0].set(title="Số trận theo tập", ylabel="Số trận")
ax[0].legend()
ax[1].bar(order, label_table.loc[order, "A thắng (%)"], color="#258a71")
ax[1].set(title="Tỷ lệ A thắng theo tập", ylabel="Tỷ lệ (%)", ylim=(0, 100))
for i,s in enumerate(order):
    ax[1].text(i, label_table.loc[s, "A thắng (%)"]+2, f"n={len(frames[s])}", ha="center")
fig.tight_layout(); plt.show()
print("Mỗi trận thuộc đúng một tập; holdout nhỏ nên tỷ lệ dao động mạnh khi thêm/bớt vài trận.")

In [ ]:
side_counts = pd.DataFrame({s: frames[s]["team_a_side"].value_counts().reindex(["blue","red"],fill_value=0) for s in SPLITS}).T
ax = side_counts.plot.bar(color=["#3b82b8", "#d25a58"], figsize=(8,3.5), rot=0, title="Phe của đội A theo tập")
ax.set(xlabel="Tập", ylabel="Số trận"); ax.legend(title="Phe A")
plt.tight_layout(); plt.show()
print("Phân bố phe chỉ phản ánh lựa chọn/ghi nhận trong mẫu, không chứng minh lợi thế phe.")

In [ ]:
patch_counts = pd.DataFrame({s: frames[s]["patch_id"].value_counts() for s in SPLITS}).T.fillna(0).astype(int)
ax = patch_counts.T.plot.bar(figsize=(9,3.8), rot=20, title="Số trận theo patch và tập")
ax.set(xlabel="Patch", ylabel="Số trận"); ax.legend(title="Tập")
plt.tight_layout(); plt.show()
print("Patch ít trận và patch mới chỉ có ở test cần được đọc cùng cỡ mẫu.")

In [ ]:
top = hero_frequency.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(top["tên tướng"], top["lượt_chọn"], color="#6f8bbd")
ax.set(xlabel="Lượt chọn (đội × trận)", ylabel="Tướng", title="Top 12 tướng được chọn nhiều nhất — toàn giải")
fig.tight_layout(); plt.show()
print("Tần suất chọn không đồng nghĩa với sức mạnh tướng; biểu đồ dùng toàn giải để mô tả, không fit mô hình.")

In [ ]:
heat = side_patch_cross.copy()
heat["nhóm"] = heat["tập"] + " / " + heat["team_a_side"].astype(str)
rates = heat.pivot(index="nhóm", columns="patch_id", values="A thắng (%)")
counts = heat.pivot(index="nhóm", columns="patch_id", values="n").reindex_like(rates)
fig, ax = plt.subplots(figsize=(max(7, 1.7*len(rates.columns)), max(3.5, .52*len(rates))))
im = ax.imshow(rates.to_numpy(dtype=float), cmap="YlGn", vmin=0, vmax=100, aspect="auto")
ax.set(xticks=range(len(rates.columns)), yticks=range(len(rates)), xticklabels=rates.columns,
       yticklabels=rates.index, xlabel="Patch", ylabel="Tập / phe đội A",
       title="Tỷ lệ A thắng (%) theo phe và patch — ô ghi % (n)")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
for i in range(len(rates)):
    for j in range(len(rates.columns)):
        n = counts.iloc[i,j]
        if pd.notna(n):
            ax.text(j,i,f"{rates.iloc[i,j]:.0f}% (n={int(n)})" + ("*" if n<10 else ""), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="A thắng (%)")
fig.tight_layout(); plt.show()
print("* n < 10: tỷ lệ quan sát rất bất định; không suy ra tác động nhân quả của phe hoặc patch.")

In [ ]:
plot_errors = pd.DataFrame({s: {
    "thiếu đầu vào":int(quality_flags[s]["thiếu_đầu_vào"].sum()),
    "JSON lỗi":int(quality_flags[s][["A_json_lỗi","B_json_lỗi"]].any(axis=1).sum()),
    "sai số lượng":int(quality_flags[s][["A_sai_số_lượng","B_sai_số_lượng"]].any(axis=1).sum()),
    "lặp trong/giữa đội":int(quality_flags[s][["A_lặp_trong_đội","B_lặp_trong_đội","trùng_giữa_hai_đội"]].any(axis=1).sum()),
    "trùng game_id":int(quality_flags[s]["game_id_trùng"].sum()),
} for s in SPLITS}).T
ax = plot_errors.T.plot.bar(figsize=(9,3.8), rot=18, title="Số dòng có lỗi theo loại và tập")
ax.set(xlabel="Loại lỗi (có thể chồng lấn)", ylabel="Số dòng")
ax.legend(title="Tập"); plt.tight_layout(); plt.show()
print("Cột bằng 0 nghĩa là không quan sát lỗi loại đó, không phải đảm bảo dữ liệu không có lỗi khác.")

## 8. Kết luận EDA và lưu ý cho tiền xử lý

Đoạn dưới đây tổng kết **tự động từ CSV** để không để lại nhận xét cũ nếu snapshot thay đổi. Các quan sát từ test chỉ là mô tả; quyết định tiền xử lý/mã hóa phải dựa trên train, validation dùng cho đánh giá trong quá trình phát triển và test chỉ đánh giá cuối cùng.

In [ ]:
total = sum(len(df) for df in frames.values())
issues = {s: int(quality_flags[s]["có_bất_thường"].sum()) for s in SPLITS}
new_text = "; ".join(f"{s}: {len(new_categories[s]['tướng mới'])} tướng, {len(new_categories[s]['patch mới'])} patch" for s in ("validation","test"))
conclusion = f"""### Kết luận mô tả
- **Quy mô:** {total} trận thuộc {sum(df['series_id'].nunique() for df in frames.values())} series của một giải; train/validation/test lần lượt {', '.join(str(len(frames[s])) for s in SPLITS)} trận. Mỗi dòng là một trận, không phải một series hay một lượt chọn.
- **Thuộc tính & nhãn:** {', '.join(features)} là đầu vào baseline; `{label}` là nhãn. Tỷ lệ A thắng train/validation/test lần lượt {', '.join(f'{label_table.loc[s, "A thắng (%)"]:.1f}%' for s in SPLITS)}. Đọc các bảng phe/patch cùng số mẫu; không suy ra nhân quả.
- **Chất lượng:** số dòng có ít nhất một bất thường (các cờ nêu trên) train/validation/test: {', '.join(str(issues[s]) for s in SPLITS)}; xem bảng theo loại để phân biệt thiếu, lỗi JSON, đội hình, trùng ID và nhãn sai. Không tự loại dòng trong EDA.
- **Tướng:** {picks['hero_id'].nunique()} tướng được chọn trong toàn giải, {len(train_heroes)} trong train. Top theo lượt chọn: {', '.join(hero_frequency.head(3)['tên tướng'])}. Tỷ lệ thắng thô có độ bất định lớn với tướng ít trận.
- **Danh mục mới so với train:** {new_text}. Giá trị mới không phải missing; danh sách ID và patch có trong bảng so sánh. Patch chỉ có test: {', '.join(sorted(set(frames['test']['patch_id'].dropna()) - set(pd.concat([frames[s]['patch_id'] for s in ('train','validation')]).dropna()))) or 'không có'}.
- **Hạn chế:** một giải, {total} trận; chia thời gian theo lịch series (proxy), chưa xác minh thời điểm thi đấu thực tế. Holdout nhỏ, không nên khái quát tỷ lệ thắng hoặc khác biệt patch/phe.
- **Hệ quả cho bước tiếp theo:** chỉ fit cách mã hóa/điền thiếu trên train; dự kiến cách xử lý danh mục tướng/patch chưa từng thấy mà không refit holdout. Xem xét và giải thích từng bất thường trước khi quyết định xử lý. ID/metadata, thống kê toàn giải và kết quả test không được dùng làm đầu vào hoặc để điều chỉnh mô hình.
"""
display(Markdown(conclusion))